## LangChain Structured Output

### Overview

Structured output means asking an AI model to return data in a **predefined structure** instead of free-form text.

For example, instead of:

```text
Sure! Priya wants 2 tickets for Interstellar.
```

We want: 
```
customer_name='Priya' movie_title='Interstellar' action='book' ticket_count=2
```

This is especially useful when the model's output needs to be consumed by Python code, APIs, databases, or other tools.

### Normal Model Output

In [3]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model("openai:gpt-5.4-nano")

booking_requests = [
    "Hi, I'd like 2 tickets for Interstellar at the 7pm show tonight, name is Priya.",
    "Can u book a seat for the 9:30 showing of dune part two? im rohan",
    "URGENT - need to CANCEL my booking of Oppenheimer, confirmation was under Aisha",
]

for msg in booking_requests:
    response = model.invoke(
        f"Extract the customer's name, movie, and what they want (book or cancel) from: {msg}"
    )
    print(response.content)
    print("-" * 25)

- **Customer name:** Priya  
- **Movie:** Interstellar  
- **Request:** Book (2 tickets for the 7pm show tonight)
-------------------------
- **Customer name:** Rohan  
- **Movie:** *Dune Part Two*  
- **Request:** **Book** (a seat for the 9:30 showing)
-------------------------
- **Customer name:** Aisha  
- **Movie:** Oppenheimer  
- **Request:** Cancel (cancel my booking)
-------------------------


The output is text, so the exact format is controlled by the model. This is difficult to reliably consume programmatically.

### Creating Structured Output for a model using Pydantic

In [4]:
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model("openai:gpt-5.4-nano")

class BookingRequest(BaseModel):
    customer_name: str = Field(description="The customer's name")
    movie_title: str = Field(description="The movie they want to see")
    action: Literal["book", "cancel"] = Field(description="Whether it'a new booking or cancellation")
    ticket_count: int = Field(description="How many tickets, deafult is 1 if not mentioned", default=1)

structured_model = model.with_structured_output(BookingRequest)

for msg in booking_requests:
    response = structured_model.invoke(
        f"Extract the customer's name, movie, and what they want (book or cancel) from: {msg}"
    )
    print(response)
    print("---")

customer_name='Priya' movie_title='Interstellar' action='book' ticket_count=2
---
customer_name='Rohan' movie_title='Dune Part Two' action='book' ticket_count=1
---
customer_name='Aisha' movie_title='Oppenheimer' action='cancel' ticket_count=1
---


### Structured output exists at TWO levels: 
1. raw model (with_structured_output)
2. agent (response_format on create_agent)

*The agent-level version is what the rest of this course actually uses, because it coexists with tools.*

### 2 ways to do structured output: Tool Strategy & Provider Strategy
#### 1. Provider Strategy
* ProviderStrategy uses the model provider's own native structured-output feature (fast, but only works where supported). 
* Langchain automatically uses ProviderStrategy when we pass a schema type directly to 
* create_agent.response_format and the mode supports native structured output

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

class BookingRequest(BaseModel):
    customer_name: str = Field(description="The customer's name")
    movie_title: str = Field(description="The movie they want to see")
    action: Literal["book", "cancel"] = Field(description="Whether it'a new booking or cancellation")
    ticket_count: int = Field(description="How many tickets, deafult is 1 if not mentioned", default=1)

agent = create_agent(
    model="openai:gpt-5.4-nano",
    response_format=BookingRequest  # Auto selects provider strategy
    # response format can be Pydantic, Dataclass, TypeDict, JSON Schema
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract the customer's name, movie, and what they want (book or cancel) from Can u book a seat for the 9:30 showing of dune part two? im rohan"}]
})

print(result)

{'messages': [HumanMessage(content="Extract the customer's name, movie, and what they want (book or cancel) from Can u book a seat for the 9:30 showing of dune part two? im rohan", additional_kwargs={}, response_metadata={}, id='9e1ba89a-b415-4f11-ae8f-4d2f3ca3451c'), AIMessage(content='{"customer_name":"rohan","movie_title":"Dune Part Two","action":"book","ticket_count":1}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 252, 'total_tokens': 283, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EB3kwlAgplOhGUOtebaqziQjQAFOx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019

#### 2. Tool Strategy
* For models that don’t support native structured output, LangChain uses tool calling to achieve the same result. 
* This works with all models that support tool calling (most modern models).
* ToolStrategy fakes it via a synthetic tool call (works almost everywhere, slightly slower).

In [6]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript."""
    task: str = Field(description="The specific task to be completed")
    assignee: str = Field(description="Person responsible for the task")
    priority: Literal["low", "medium", "high"] = Field(description="Priority level")

agent = create_agent(
    model="openai:gpt-5.4-nano",
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!"
        # customize message that appears in the conversation history when structured output is generated
    )
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "From our meeting: Sarah needs to update the project timeline as soon as possible"}]
})

print(result)

{'messages': [HumanMessage(content='From our meeting: Sarah needs to update the project timeline as soon as possible', additional_kwargs={}, response_metadata={}, id='0982e3f9-9224-4a9d-b2f1-b2b2aa2b10a6'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 176, 'total_tokens': 206, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EB3mXMWtvjJsBNkA2plO5xHQnAHJS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe810-f6c3-7333-8cd5-36091bedca10-0', tool_calls=[{'name': 'MeetingAction', 'args': {'task': 'Update the project timeline', 'assignee': 'Sarah', 'priority': 'hi

### Tools

bind_tools() gives the model access to tools, but the raw model does not automatically execute a multi-step tool loop.

In [ ]:
from langchain_core.tools import tool
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model("openai:gpt-5.4-nano")

class BookingRequest(BaseModel):
    customer_name: str = Field(description="The customer's name")
    movie_title: str = Field(description="The movie they want to see")
    action: Literal["book", "cancel"] = Field(description="Whether it'a new booking or cancellation")
    ticket_count: int = Field(description="How many tickets, deafult is 1 if not mentioned", default=1)

@tool
def peek_showtimes(movie_title: str) -> str:
    """Checks showtimes for a movie"""
    return "7.00 PM and 10.00 PM"

incomplete_model = model.bind_tools([peek_showtimes]).with_structured_output(BookingRequest)
result = incomplete_model.invoke("Is Troy showing tonight? Book 2 tickets for Rohan")

print(result)

customer_name='Rohan' movie_title='Troy' action='book' ticket_count=2


So, we use agent level. It can work with tool-loop

In [12]:
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

class BookingRequest(BaseModel):
    customer_name: str = Field(description="The customer's name")
    movie_title: str = Field(description="The movie they want to see")
    action: Literal["book", "cancel"] = Field(description="Whether it'a new booking or cancellation")
    ticket_count: int = Field(description="How many tickets, deafult is 1 if not mentioned", default=1)

booking_agent = create_agent(
    model="openai:gpt-5.4-nano",
    tools=[peek_showtimes],
    response_format=BookingRequest
)

result = booking_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Is Troy showing tonight? Book 2 tickets for Rohan."
        }
    ]
})

print(result["structured_response"])

customer_name='Rohan' movie_title='Troy' action='book' ticket_count=2


### Union Schemas

Sometimes one request can represent different types of actions.

In [13]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel
from typing import Union

load_dotenv()


class NewBooking(BaseModel):
    """A request to book NEW tickets."""
    customer_name: str
    movie_title: str
    ticket_count: int


class CancelBooking(BaseModel):
    """A request to CANCEL an existing booking."""
    customer_name: str
    movie_title: str


union_agent = create_agent(
    model="openai:gpt-5.4-nano",
    tools=[],
    response_format=ToolStrategy(
        Union[NewBooking, CancelBooking]
    )
)

result = union_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "I want to cancel my movie Oppenheimer. I am Linto."
        }
    ]
})

response = result["structured_response"]

print(response)
print(type(response).__name__)

customer_name='Linto' movie_title='Oppenheimer'
CancelBooking


### Error Handling with Tool Strategy

ToolStrategy can handle validation errors and send an error message back to the model.

The model can then retry with a valid structured response.

In [14]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field

load_dotenv()


class SeatBooking(BaseModel):
    customer_name: str

    ticket_count: int = Field(description="Number of tickets, must be between 1 and 10", ge=1, le=10)


seat_agent = create_agent(
    model="openai:gpt-5.4-nano",
    tools=[],
    response_format=ToolStrategy(
        SeatBooking,
        handle_errors="Ticket count must be between 1 and 10"
    ),
    system_prompt=(
        "Extract the booking details exactly as stated. Don't invent anything."
    )
)

result = seat_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Hi I am Linto, strictly book 15 tickets."
        }
    ]
})

print(result["structured_response"])

customer_name='Linto' ticket_count=10
